# Simple pendulum beyond small angles

*mechanics · beginner* · [open in Tensorizer](https://jackyko1991.github.io/tensorizer/#/mechanics/simple_pendulum)

The full nonlinear pendulum equation for several release amplitudes at once. All pendulums live in one batched tensor, and each is compared with the small-angle cosine you learn in school.

**Vector form**

$$\ddot\theta = -\frac{g}{L}\,\sin\theta$$

**Index (tensor) form**

$$\ddot\theta_b = -\frac{g}{L}\,\sin\theta_b,\qquad \theta_b^{\text{small}}(t) = \theta_b(0)\cos\!\big(\sqrt{g/L}\,t\big)$$

| Symbol | Code | Shape | Meaning |
|---|---|---|---|
| $\theta_b$ | `y[:, 0]` | (B,) | angle of pendulum $b$ from the vertical |
| $\omega_b = \dot\theta_b$ | `y[:, 1]` | (B,) | angular velocity of pendulum $b$ |
| $\ddot\theta_b$ | `angular_acceleration(theta, g, L)` | (B,) | right-hand side: $-(g/L)\sin\theta$ for every pendulum at once |
| $\theta_b(0)\cos(\Omega t)$ | `einsum('t,b->tb', cos(Ω t), theta0)` | (T, B) | small-angle solution: an outer product, no index summed |
| $E_b$ | `energy(y, g, L)` | (B,) | energy per unit mass $\tfrac12L^2\omega^2 + gL(1 - \cos\theta)$, conserved |

For small swings $\sin\theta \approx \theta$ and every pendulum has the same period $2\pi\sqrt{L/g}$. Bigger swings take longer: the exact period is $4\sqrt{L/g}\,K(\sin(\theta_0/2))$ with the complete elliptic integral $K$, about 18% longer at 90°. Watch the large-amplitude pendulum fall behind its dashed small-angle partner.


## Setup

On Colab, choose **Runtime → Change runtime type → GPU** to run on a GPU — the code uses it automatically.

In [ ]:
import os
import subprocess
import sys

# Shared helpers (problems/_common) come from the repository; skip the clone if we are already inside it.
if not os.path.isdir("problems"):
    if not os.path.isdir("tensorizer"):
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/jackyko1991/tensorizer.git"], check=True)
    os.chdir("tensorizer")
sys.path.insert(0, os.getcwd())
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pyyaml", "tqdm"], check=True)

import torch

print("torch", torch.__version__, "· device:", "cuda" if torch.cuda.is_available() else "cpu")


## 1. `main.py` — main loop

In [ ]:
%%writefile problems/mechanics/simple_pendulum/main.py
"""Main loop: parameters → tensors → solve → plot."""

import math
from pathlib import Path

from problems._common.runtime import cli, device

from . import equation, plot, solve, tensors


def run(params: dict, out_dir: Path) -> Path:
    dev = device()
    g, L = params["g"], params["length"]
    theta0 = tensors.amplitudes(params["n_pendulums"], params["max_amplitude"])
    y0 = tensors.initial_state(theta0, dev)

    times, traj = solve.solve(y0, g, L, params["t_max"])
    small = equation.small_angle(y0[:, 0], times, g, L)  # (T, B)

    measured = solve.periods(times, traj[:, :, 0])
    exact = equation.exact_period(y0[:, 0], g, L)
    print(f"small-angle period 2π√(L/g) = {2 * math.pi * math.sqrt(L / g):.3f} s")
    for deg, tm, te in zip(theta0.rad2deg().tolist(), measured.tolist(), exact.tolist()):
        print(f"amplitude {deg:5.1f}°  period {tm:6.3f} s  (elliptic integral {te:6.3f} s)")
    return plot.animate(times, traj, small, L, out_dir / "simple_pendulum.gif")


if __name__ == "__main__":
    cli(run, __file__)


## 2. `equation.py` — physics → tensor translation

In [ ]:
%%writefile problems/mechanics/simple_pendulum/equation.py
r"""Physics → tensor translation.

A point mass on a light rod of length L swings under gravity g. Newton's law along the arc gives

    \ddot\theta = -\frac{g}{L} \sin\theta .

We release B pendulums from different amplitudes and keep them in one tensor, so the
batch index b labels the pendulum:

    \ddot\theta_b = -\frac{g}{L} \sin\theta_b

Nothing is summed here: every pendulum obeys the same law independently, and PyTorch
applies sin to all B angles at once. As a first-order system with state y = (θ, ω):

    \dot\theta_b = \omega_b,    \dot\omega_b = -\frac{g}{L} \sin\theta_b

For small angles sin θ ≈ θ and the solution is θ_b(t) = θ_b(0) cos(Ω t) with Ω = √(g/L).
"""

import math

import torch


def angular_acceleration(theta: torch.Tensor, g: float, L: float) -> torch.Tensor:
    r"""\ddot\theta_b = -(g/L) \sin\theta_b.   theta: (B,) → (B,)"""
    return -(g / L) * torch.sin(theta)


def rhs(y: torch.Tensor, g: float, L: float) -> torch.Tensor:
    """dy/dt for the state y of shape (B, 2): y[:, 0] = angle θ, y[:, 1] = angular velocity ω."""
    theta_dot = y[:, 1]
    omega_dot = angular_acceleration(y[:, 0], g, L)
    return torch.stack([theta_dot, omega_dot], dim=1)


def small_angle(theta0: torch.Tensor, t: torch.Tensor, g: float, L: float) -> torch.Tensor:
    r"""Linearised solution θ_{tb} = θ_b(0) cos(Ω t_t).   theta0: (B,), t: (T,) → (T, B)"""
    omega = math.sqrt(g / L)
    return torch.einsum("t,b->tb", torch.cos(omega * t), theta0)  # outer product, nothing summed


def energy(y: torch.Tensor, g: float, L: float) -> torch.Tensor:
    r"""Energy per unit mass E_b = ½ L² ω_b² + g L (1 − cos θ_b).   (..., B, 2) → (..., B)"""
    theta, omega = y[..., 0], y[..., 1]
    return 0.5 * L**2 * omega**2 + g * L * (1 - torch.cos(theta))


def exact_period(theta0: torch.Tensor, g: float, L: float) -> torch.Tensor:
    r"""T_b = 4 √(L/g) K(sin(θ_b/2)), with the complete elliptic integral K from the
    arithmetic–geometric mean:  K(k) = π / (2 AGM(1, √(1 − k²))).   (B,) → (B,)"""
    k = torch.sin(theta0 / 2)
    a, b = torch.ones_like(k), torch.sqrt(1 - k**2)
    for _ in range(10):  # the AGM converges quadratically: 10 rounds is plenty
        a, b = (a + b) / 2, torch.sqrt(a * b)
    return 4 * math.sqrt(L / g) * math.pi / (2 * a)


## 3. `tensors.py` — PyTorch tensors

In [ ]:
%%writefile problems/mechanics/simple_pendulum/tensors.py
"""Build the concrete tensors: the release amplitudes and the batched initial state."""

import torch


def amplitudes(n: int, max_deg: float) -> torch.Tensor:
    """n release angles from 10° up to max_deg (or just max_deg for a single pendulum), in radians: (B,)."""
    if n == 1:
        return torch.deg2rad(torch.tensor([max_deg]))
    return torch.deg2rad(torch.linspace(10.0, max_deg, n))


def initial_state(theta0: torch.Tensor, device: torch.device) -> torch.Tensor:
    """y0 of shape (B, 2): every pendulum is released from rest.

    y0[b, 0] = θ_b(0)     release angle
    y0[b, 1] = 0          angular velocity ω_b(0)
    """
    theta0 = theta0.to(device)
    return torch.stack([theta0, torch.zeros_like(theta0)], dim=1)


## 4. `solve.py` — solver

In [ ]:
%%writefile problems/mechanics/simple_pendulum/solve.py
"""Integrate all pendulums together with 4th-order Runge–Kutta and measure their periods."""

import torch

from problems._common.integrators import integrate

from .equation import rhs


def solve(y0: torch.Tensor, g: float, L: float, t_max: float, dt: float = 0.005) -> tuple[torch.Tensor, torch.Tensor]:
    """Return (times (T,), trajectory (T, B, 2)). One RK4 step updates all B pendulums."""
    n_steps = int(round(t_max / dt))
    traj = integrate(lambda y, t: rhs(y, g, L), y0, dt, n_steps)  # (T, B, 2)
    times = torch.arange(n_steps + 1, device=y0.device) * dt
    return times, traj


def periods(times: torch.Tensor, theta: torch.Tensor) -> torch.Tensor:
    """Period of each pendulum from its first downward zero crossing: (T, B) → (B,).

    Released from rest at θ > 0, a pendulum reaches θ = 0 after exactly a quarter period.
    We find the step k where θ changes sign and interpolate linearly between k and k + 1.
    """
    crossed = (theta[:-1] > 0) & (theta[1:] <= 0)  # (T-1, B)
    k = crossed.int().argmax(dim=0)  # (B,) first crossing step
    b = torch.arange(theta.shape[1])
    before, after = theta[k, b], theta[k + 1, b]
    dt = times[1] - times[0]
    quarter = times[k] + dt * before / (before - after)
    return 4 * quarter


## 5. `plot.py` — 2D/3D plot → GIF

In [ ]:
%%writefile problems/mechanics/simple_pendulum/plot.py
"""Animate the pendulums swinging side by side, with θ(t) traces against the small-angle cosine."""

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch

from problems._common.plotting import frame_indices, save_gif


def animate(times: torch.Tensor, traj: torch.Tensor, small: torch.Tensor, L: float, path: Path) -> Path:
    t = times.cpu().numpy()  # (T,)
    theta = traj[:, :, 0].cpu().numpy()  # (T, B)
    theta_small = small.cpu().numpy()  # (T, B)
    n = theta.shape[1]
    pivots = 2.2 * L * np.arange(n)  # x position of each pivot

    fig, (ax_swing, ax_trace) = plt.subplots(2, 1, figsize=(7, 4.6), gridspec_kw={"height_ratios": [1.1, 1]})

    # --- top: the pendulums ---
    ax_swing.set_xlim(-1.2 * L, pivots[-1] + 1.2 * L)
    ax_swing.set_ylim(-1.15 * L, 1.35 * L)
    ax_swing.set_aspect("equal", adjustable="box")
    ax_swing.set_xticks([])
    ax_swing.set_yticks([])
    ax_swing.grid(False)
    ax_swing.plot(pivots, np.zeros(n), "k^", ms=6)
    rods = [ax_swing.plot([], [], color="#6b7280", lw=1.5)[0] for _ in range(n)]
    bobs = [ax_swing.plot([], [], "o", ms=9)[0] for _ in range(n)]
    for b in range(n):
        label = f"{np.degrees(theta[0, b]):.0f}°"
        ax_swing.text(pivots[b] + 0.15 * L, 0.1 * L, label, fontsize=8, color=bobs[b].get_color())
    clock = ax_swing.text(0.02, 0.97, "", transform=ax_swing.transAxes, va="top", fontsize=9)

    # --- bottom: θ(t), solid = full equation, dashed = small-angle solution ---
    ax_trace.set_xlim(t[0], t[-1])
    ax_trace.set_ylim(-np.degrees(np.abs(theta).max()) * 1.1, np.degrees(np.abs(theta).max()) * 1.1)
    ax_trace.set_xlabel("t (s)")
    ax_trace.set_ylabel("θ (°)")
    ax_trace.set_title("solid: full equation (sin θ)    dashed: small-angle (sin θ ≈ θ)", fontsize=9)
    traces = [ax_trace.plot([], [], lw=1.4, color=bob.get_color())[0] for bob in bobs]
    linear = [ax_trace.plot([], [], lw=0.9, ls="--", color=bob.get_color())[0] for bob in bobs]

    def update(step: int):
        for b in range(n):
            x_bob = pivots[b] + L * np.sin(theta[step, b])
            y_bob = -L * np.cos(theta[step, b])
            rods[b].set_data([pivots[b], x_bob], [0.0, y_bob])
            bobs[b].set_data([x_bob], [y_bob])
            traces[b].set_data(t[: step + 1], np.degrees(theta[: step + 1, b]))
            linear[b].set_data(t[: step + 1], np.degrees(theta_small[: step + 1, b]))
        clock.set_text(f"t = {t[step]:.2f} s")

    fig.tight_layout()
    return save_gif(fig, update, frame_indices(len(t), 100), path)


## Run

Change the parameters (or any file above) and run again.

In [ ]:
import importlib
from pathlib import Path

from IPython.display import Image, display

from problems._common.runtime import load_meta, resolve_params

params = {
    'length': 1.0,
    'g': 9.81,
    'n_pendulums': 4,
    'max_amplitude': 150.0,
    't_max': 6.0
}

# Re-import so edits in the %%writefile cells above take effect.
for name in [m for m in sys.modules if m.startswith("problems.mechanics.simple_pendulum")]:
    del sys.modules[name]
main = importlib.import_module("problems.mechanics.simple_pendulum.main")
problem_dir = Path("problems/mechanics/simple_pendulum")
output = main.run(resolve_params(load_meta(problem_dir), params), Path("out"))
display(Image(filename=str(output)))
